In [12]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 2.3.3
numpy version: 2.0.2


In [13]:
import numpy as np

def gini_coefficient(incomes):
    """
    Computes the Gini coefficient for a list/array of incomes.
    0 = perfect equality, 1 = perfect inequality.
    """
    incomes = np.array(incomes)
    n = len(incomes)
    mean_income = np.mean(incomes)
    
    # sum of |x_i - x_j| over all pairs
    total_diff = 0
    for i in range(n):
        for j in range(n):
            total_diff += abs(incomes[i] - incomes[j])
    
    gini = total_diff / (2 * n**2 * mean_income)
    return gini

    # everyone has the same income -> should print 0.0
print(gini_coefficient([50, 50, 50, 50]))

# one person has everything, rest have nothing 
# should print a number close to 1
print(gini_coefficient([1, 1, 1, 1000]))

# mixed example
print(gini_coefficient([50, 50, 70, 70, 70, 90, 150, 150, 150, 150]))

0.0
0.7470089730807578
0.226


In [14]:
def spectral_gap(transition_matrix):
    """
    Computes the spectral gap of a transition matrix.
    Spectral gap = 1 - second largest eigenvalue.
    Close to 1 = fast mixing (mobile). Close to 0 = slow mixing (immobile).
    """
    matrix = np.array(transition_matrix)
    eigenvalues = np.linalg.eigvals(matrix)
    
    # sort eigenvalues by absolute value, descending
    sorted_eigs = sorted(eigenvalues, key=abs, reverse=True)
    
    largest = sorted_eigs[0]
    second_largest = sorted_eigs[1]
    
    gap = 1 - abs(second_largest)
    return gap, second_largest

sticky_matrix = [
    [0.6, 0.3, 0.1],
    [0.2, 0.6, 0.2],
    [0.1, 0.3, 0.6]
]

gap, second_eig = spectral_gap(sticky_matrix)
print("Second eigenvalue:", second_eig)
print("Spectral gap:", gap)

mobile_matrix = [
    [0.34, 0.33, 0.33],
    [0.33, 0.34, 0.33],
    [0.33, 0.33, 0.34]
]

gap2, second_eig2 = spectral_gap(mobile_matrix)
print("Second eigenvalue:", second_eig2)
print("Spectral gap:", gap2)

Second eigenvalue: 0.4999999999999997
Spectral gap: 0.5000000000000002
Second eigenvalue: 0.010000000000000004
Spectral gap: 0.99


In [15]:
def simulate_population(transition_matrix, initial_distribution, generations, income_values):
    """
    Simulates a population moving through income groups over multiple generations.
    Returns the income distribution (as individual income values) at each generation.
    """
    matrix = np.array(transition_matrix)
    distribution = np.array(initial_distribution) 
    
    history = []
    for gen in range(generations):
        # distribution is fraction of population in each group
        history.append(distribution.copy())
        distribution = distribution @ matrix  # one generation's transition
    
    return history

initial = [1.0, 0.0, 0.0]  # everyone starts in low income
history = simulate_population(sticky_matrix, initial, generations=10, income_values=[30, 60, 120])

for i, dist in enumerate(history):
    print(f"Generation {i}: {dist}")

Generation 0: [1. 0. 0.]
Generation 1: [0.6 0.3 0.1]
Generation 2: [0.43 0.39 0.18]
Generation 3: [0.354 0.417 0.229]
Generation 4: [0.3187 0.4251 0.2562]
Generation 5: [0.30186 0.42753 0.27061]
Generation 6: [0.293683 0.428259 0.278058]
Generation 7: [0.2896674 0.4284777 0.2818549]
Generation 8: [0.28768147 0.42854331 0.28377522]
Generation 9: [0.28669507 0.42856299 0.28474194]


In [16]:
def distribution_to_incomes(distribution, income_values, population_size=10000):
    """
    Converts a group distribution (fractions) into an actual list of individual incomes,
    so we can compute Gini on it.
    """
    incomes = []
    for fraction, income in zip(distribution, income_values):
        count = int(round(fraction * population_size))
        incomes.extend([income] * count)
    return incomes

In [17]:
final_distribution = history[-1]  # last generation 
income_values = [30, 60, 120]

incomes = distribution_to_incomes(final_distribution, income_values)
steady_state_gini = gini_coefficient(incomes)

print("Steady-state distribution:", final_distribution)
print("Steady-state Gini coefficient:", steady_state_gini)

Steady-state distribution: [0.28669507 0.42856299 0.28474194]
Steady-state Gini coefficient: 0.2680137249748106


In [18]:
history_mobile = simulate_population(mobile_matrix, initial, generations=20, income_values=[30, 60, 120])
final_dist_mobile = history_mobile[-1]

incomes_mobile = distribution_to_incomes(final_dist_mobile, income_values)
gini_mobile = gini_coefficient(incomes_mobile)

print("Mobile matrix steady-state distribution:", final_dist_mobile)
print("Mobile matrix Gini:", gini_mobile)
print("Mobile matrix spectral gap:", spectral_gap(mobile_matrix)[0])
print("---")
print("Sticky matrix Gini:", steady_state_gini)
print("Sticky matrix spectral gap:", spectral_gap(sticky_matrix)[0])

Mobile matrix steady-state distribution: [0.33333333 0.33333333 0.33333333]
Mobile matrix Gini: 0.2857142857142857
Mobile matrix spectral gap: 0.99
---
Sticky matrix Gini: 0.2680137249748106
Sticky matrix spectral gap: 0.5000000000000002
